# 03 — Controlled Noise Benchmarks

This notebook establishes the **noise-sensitivity baseline** for the Adaptive_QEM_IBM study before real IBM Quantum execution. The same benchmark suite is evaluated under controlled readout, gate, and combined noise models.

**Flow:** ideal simulation → controlled noisy simulation → IBM hardware → quantum error mitigation.

## Noise Scenarios

| Scenario | Main source | Purpose |
|---|---|---|
| Ideal | None | Reference baseline |
| Readout-only | Measurement assignment error | Measurement sensitivity |
| Gate-only | 1Q/2Q depolarizing + thermal relaxation | Circuit/gate sensitivity |
| Combined | Gate + readout noise | Realistic noisy baseline |

The parameters below are controlled simulation values, not IBM calibration values.

In [ ]:
from pathlib import Path
import sys, json, math
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data" / "noisy"
DATA_DIR.mkdir(parents=True, exist_ok=True)
SHOTS = 4096
SEED_SIMULATOR = 42
print("Project root:", PROJECT_ROOT)


In [ ]:
from circuits.bell import bell_phi_plus
from circuits.ghz import create_ghz
from circuits.teleportation import teleportation
from circuits.superdense import superdense_coding
from circuits.qft import qft
from circuits.grover import grover_2qubit
from circuits.qaoa import qaoa_two_node
from circuits.qpe import qpe
from circuits.qrng import qrng

benchmarks = {
    "Bell_Phi_Plus": bell_phi_plus(),
    "GHZ_3": create_ghz(3), "GHZ_4": create_ghz(4), "GHZ_5": create_ghz(5),
    "Teleportation": teleportation(),
    "Superdense_00": superdense_coding("00"), "Superdense_01": superdense_coding("01"),
    "Superdense_10": superdense_coding("10"), "Superdense_11": superdense_coding("11"),
    "QFT_3": qft(3), "QFT_4": qft(4), "Grover_2Q": grover_2qubit(),
    "QAOA_2Q": qaoa_two_node(gamma=math.pi/4, beta=math.pi/8),
    "QPE": qpe(), "QRNG_4": qrng(4)
}
print(f"Loaded {len(benchmarks)} benchmark circuits.")


## 1. Controlled Noise Parameters

In [ ]:
NOISE_CONFIG = {
    "readout_error": 0.02,
    "depolarizing_1q": 0.001,
    "depolarizing_2q": 0.01,
    "thermal_t1_us": 100.0,
    "thermal_t2_us": 80.0,
    "thermal_1q_gate_us": 0.05,
    "thermal_2q_gate_us": 0.30,
}
NOISE_CONFIG


## 2. Build Noise Models

In [ ]:
from qiskit_aer.noise import NoiseModel, ReadoutError, depolarizing_error, thermal_relaxation_error

def readout_model(p):
    m=NoiseModel(); r=ReadoutError([[1-p,p],[p,1-p]]); m.add_all_qubit_readout_error(r); return m

def gate_model(c):
    m=NoiseModel()
    e1=depolarizing_error(c["depolarizing_1q"],1); e2=depolarizing_error(c["depolarizing_2q"],2)
    one=["h","x","y","z","sx","rx","ry","rz"]; two=["cx","cz","swap"]
    m.add_all_qubit_quantum_error(e1,one); m.add_all_qubit_quantum_error(e2,two)
    t1=thermal_relaxation_error(c["thermal_t1_us"],c["thermal_t2_us"],c["thermal_1q_gate_us"])
    t2a=thermal_relaxation_error(c["thermal_t1_us"],c["thermal_t2_us"],c["thermal_2q_gate_us"])
    t2=t2a.tensor(t2a)
    m.add_all_qubit_quantum_error(t1,one); m.add_all_qubit_quantum_error(t2,two)
    return m

def combined_model(c):
    m=gate_model(c); p=c["readout_error"]; r=ReadoutError([[1-p,p],[p,1-p]]); m.add_all_qubit_readout_error(r); return m

noise_models={"readout_only":readout_model(NOISE_CONFIG["readout_error"]),"gate_only":gate_model(NOISE_CONFIG),"combined":combined_model(NOISE_CONFIG)}
print("Noise models created:", list(noise_models))


## 3. Simulation Helpers

In [ ]:
from qiskit_aer import AerSimulator

def run(qc, noise_model=None):
    sim=AerSimulator(noise_model=noise_model)
    return sim.run(qc,shots=SHOTS,seed_simulator=SEED_SIMULATOR).result().get_counts()

def p_states(counts, states): return sum(counts.get(s,0) for s in states)/SHOTS

def success(name, counts):
    if name=="Bell_Phi_Plus": return p_states(counts,["00","11"])
    if name.startswith("GHZ_"):
        n=int(name.split("_")[1]); return p_states(counts,["0"*n,"1"*n])
    if name.startswith("Superdense_"): return counts.get(name.split("_")[1],0)/SHOTS
    if name=="Grover_2Q": return counts.get("11",0)/SHOTS
    return float("nan")

def characterize(qc):
    o=qc.count_ops(); return {"qubits":qc.num_qubits,"depth":qc.depth(),"size":qc.size(),"cx":o.get("cx",0),"cz":o.get("cz",0),"swap":o.get("swap",0)}


## 4. Run Ideal and Noisy Experiments

In [ ]:
scenario_models={"ideal":None,**noise_models}
rows=[]; all_counts={}
for scenario,model in scenario_models.items():
    print("Running:",scenario)
    for name,qc in benchmarks.items():
        counts=run(qc,model); all_counts[f"{scenario}__{name}"]=counts
        r={"scenario":scenario,"circuit":name,"shots":SHOTS,"observed_states":len(counts),"most_likely_state":max(counts,key=counts.get),"most_likely_probability":max(counts.values())/SHOTS,"benchmark_success_probability":success(name,counts)}
        r.update(characterize(qc)); rows.append(r)
results=pd.DataFrame(rows)
print("Completed",len(results),"experiments.")
results.head()


## 5. Ideal vs Noisy Degradation

In [ ]:
comparison=results.pivot_table(index="circuit",columns="scenario",values="benchmark_success_probability",aggfunc="first").reset_index()
for s in ["readout_only","gate_only","combined"]:
    if s in comparison.columns: comparison[s+"_degradation"]=comparison["ideal"]-comparison[s]
comparison


## 6. Combined-Noise Sensitivity

In [ ]:
combined=results[results.scenario=="combined"].copy()
combined["error_probability"]=1-combined["benchmark_success_probability"]
combined[["circuit","qubits","depth","size","cx","cz","swap","benchmark_success_probability","error_probability"]].sort_values("depth",ascending=False)


## 7. Bell Two-Qubit Noise Sweep

In [ ]:
from qiskit_aer.noise import NoiseModel, depolarizing_error
sweep=[]
for p2 in [0.0,0.0025,0.005,0.01,0.02,0.03,0.05]:
    m=NoiseModel(); m.add_all_qubit_quantum_error(depolarizing_error(0.001,1),["h","x","z"]); m.add_all_qubit_quantum_error(depolarizing_error(p2,2),["cx","cz","swap"])
    c=run(benchmarks["Bell_Phi_Plus"],m); s=p_states(c,["00","11"])
    sweep.append({"two_qubit_depolarizing_probability":p2,"success_probability":s,"error_probability":1-s})
bell_sweep=pd.DataFrame(sweep); bell_sweep


## 8. Save Results

In [ ]:
results.to_csv(DATA_DIR/"noisy_benchmark_results.csv",index=False)
comparison.to_csv(DATA_DIR/"noisy_vs_ideal_comparison.csv",index=False)
bell_sweep.to_csv(DATA_DIR/"bell_noise_sweep.csv",index=False)
(DATA_DIR/"noisy_counts.json").write_text(json.dumps(all_counts,indent=2),encoding="utf-8")
(DATA_DIR/"noise_configuration.json").write_text(json.dumps(NOISE_CONFIG,indent=2),encoding="utf-8")
print("Saved noisy benchmark dataset to",DATA_DIR)


## Research Interpretation

This stage establishes **controlled noise sensitivity only**. It does not yet determine which QEM method is preferable. The next stage will use actual IBM Quantum backend calibration and hardware measurements.

## Next Step

Proceed to **`04_hardware_benchmarks.ipynb`** to execute the same benchmark suite on real IBM Quantum hardware and record backend calibration, transpilation, job IDs, raw counts, and hardware performance.